## Simple gen Ai APP using Langchain

In [16]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
# LanSmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="True"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [17]:
## Data Ingestion-- From the website we need to scrape the data

from langchain_community.document_loaders import WebBaseLoader

In [18]:
loader=WebBaseLoader("https://docs.langchain.com/langsmith/administration-overview#organizations")
loader

In [19]:
docs=loader.load()

In [20]:
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview#organizations', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content='Overview - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationAccount administrationOverviewGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusOn this pageResource hierarchyOrganizationsWorkspacesApplicationsResourcesAdditional infoResource tagsUser management and RBACUsersAPI keysExpir

In [21]:
### Load Data -->Docs--> Divide our text into chunks-->text-->vextors-->Embedding-->Vectore store
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)

In [22]:
documents

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview#organizations', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content='Overview - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationAccount administrationOverviewGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusOn this pageResource hierarchyOrganizationsWorkspacesApplicationsResourcesAdditional infoResource tagsUser management and RBACUsersAPI keysExpir

In [23]:
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings()

In [24]:
from langchain_community.vectorstores import FAISS
vecroestortedb=FAISS.from_documents(documents,embeddings)

In [25]:
vecroestortedb

In [26]:
## Query from a vectoe db
query="Trace requests, evaluate outputs, test prompts,"

In [27]:
result=vecroestortedb.similarity_search(query)
result[0].page_content

'\u200bBest practices\n\u200bEnvironment separation\nUse resource tags to organize resources by environment using the default tag key Environment and different values for the environment (e.g., dev, staging, prod). We do not recommend using separate workspaces for environment separation because resources cannot be shared across workspaces, which would prevent you from promoting resources (like prompts) between environments.\nResource tags vs. commit tags for prompt managementWhile both types of tags can use environment terminology like dev, staging, and prod, they serve different purposes:\nResource tags (Environment: prod): Use these to organize and filter resources across your workspace. Apply resource tags to tracing projects, datasets, and other resources (including prompts) to group them by environment, which enables filtering in the UI.'

In [28]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o")
print(llm)

output_version=None profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True} client=<openai.resources.chat.completions.completions.Completions object at 0x00000270E80664A0> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000270E80661A0> root_client=<openai.OpenAI object at 0x00000270E8064730> root_async_client=<openai.AsyncOpenAI object at 0x00000270E80654E0> model_name='gpt-4o' model_kwargs={} openai_api_key=SecretStr('***

In [29]:
## Retrieval Chain, Document chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>


"""
)

document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
| ChatOpenAI(output_version=None, profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structu

In [33]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"When you log in for the first time, a personal organization",
    "context":[Document(page_content="When you log in for the first time, a personal organization will be created for you automatically. If you’d like to collaborate with others, you can create a separate organization and invite your team members to join. There are a few important differences between your personal organization and shared organizations:")]
})


'When you log in for the first time, a personal organization is automatically created for you. If you want to collaborate with others, you can create a separate shared organization and invite team members to join. There are important differences between personal organizations and shared organizations, though the specific differences are not detailed in the provided context.'

In [34]:
### Retriver
vecroestortedb